1. Decompose into $k_v\times k_s \times k_o$ core and $V_f \times k_f$ factors
2. For all OOV $x$ across verbs, subjects, objects:
    1. For OOV $o_x$: find $(v, s, o_x)$ where all but the $o_x$ entry is included in the vocabulary. Given vocabularies $V_v, V_s, V_o$ and `excluded_role_vector` function $f_{ex}(v, s)\to o'$:
 		$$\forall (v,s,o_x)| v \in V_v;\quad s\in V_s;\quad o_x\notin V_o:$$
 	$$o_x' = f_{ex}(v, s)$$

 	2. Calculate $o'$ across all relevant $(v \in V_v,s \in V_s,o_x)$ cases and **average** over expected activations, add to $V_o$ extension matrix  $V_o' \leftarrow V_o\cup \{o'\}$

3. If no more items can be calculated with the original $V_v, V_s, V_o$ replace $V_v \leftarrow V'_v;\quad V_s \leftarrow V'_s;\quad V_o \leftarrow V'_o$ and repeat. The new items fit into the core as they live in the same k activation space


In [1]:
import numpy as np

from tucker_tensor import TuckerDecomposition

dataset = "fineweb-en"
random_state = 1
tucker = TuckerDecomposition.load_from_disk(
    dataset=dataset,
    method="siiShifted",
    divergence="fr",
    dims=1000,
    rank=150,
    iterations=500
)


2026-01-13 11:43:57.706163: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from utils import DATA_DIR
from similarity import load_eval_sentences_cached
import os

vector_path = os.path.join(DATA_DIR, "vectors", "fineweb_english_vectors.csv")
sentence_sample = load_eval_sentences_cached(vector_path=vector_path,
                                             dataset=dataset,
                                             seed=random_state,
                                             n_samples=10_000,
                                             )

In [3]:
from tqdm import tqdm
clean_sample = []
for vector in tqdm(sentence_sample):
    cleaned = []
    for i, element in enumerate(vector):
        if element == "~":
            cleaned.append("None")
        else:
            cleaned.append(element)
    clean_sample.append(tuple(cleaned))


100%|██████████| 10000/10000 [00:00<00:00, 1385996.96it/s]


In [4]:
from collections import defaultdict
out_v = defaultdict(list)
for vector in tqdm(sentence_sample):

    if vector[0] in tucker.vocab["vocab_v"]:
        continue
    elif (vector[1] in tucker.vocab["vocab_s"]) and (vector[2] in tucker.vocab["vocab_o"]):
        out_v[vector[0]].append(vector)
        print(f"{vector}")
v_extension = {}
for verb, vectors in out_v.items():
    reps = []
    for vector in vectors:
        reps.append(tucker.predicted_role_vector(vector, "verb"))
    representation = sum(reps) / len(reps)
    v_extension[verb] = representation

100%|██████████| 10000/10000 [00:00<00:00, 455962.08it/s]


('demystify', 'he', 'form')
('reclassify', 'Service', 'city')
('inject', 'I', 'something')
('availableness', 'you', 'slot')
('recap', 'I', 'sign')
('inherit', 'they', 'condition')
('grip', 'you', 'tool')
('ruin', 'this', 'taste')
('’ve', 'you', 'year')
('configuration', 'you', 'program')
('cheer', 'we', 'ourselves')
('pace', 'I', 'room')
('subsidize', 'plan', 'cost')
('relay', 'we', 'update')
('toggle', 'we', 'they')
('breed', 'health', 'thought')
('kit', 'you', 'kitchen')
('manipulate', 'we', 'degree')
('outweigh', 'value', 'impact')
('comprehend', 'article', 'threat')
('weary', 'I', 'you')
('revisit', 'word', 'idea')
('setup', 'I', 'environment')
('spray', 'use', 'water')
('profile', 'we', 'woman')
('reset', 'it', 'itself')
('despise', 'he', 'those')
('resize', 'you', 'note')
('voice', 'Russia', 'concern')
('pende', 'charge', 'outcome')
('recap', 'we', 'process')
('redirect', 'bill', 'contract')
('glean', 'team', 'insight')
('stake', 'you', 'claim')
('inherit', 'she', 'class')
('lodg

In [5]:
from tucker_tensor import _role_index, _voc_index
def extend_vocab(role, sample, tucker):
    """
    Build representations for OOV tokens in the given role ("verb", "subject", "object")
    by averaging tucker.predicted_role_vector(...) across all contexts where the other
    two roles are in-vocab.

    Parameters
    ----------
    role : {"verb","subject","object"}
    sample : iterable of (verb, subject, object) triples (strings)
    tucker : TuckerDecomposition

    Returns
    -------
    extension : dict[str, np.ndarray]
        Mapping from OOV token -> averaged predicted role vector.
    out_contexts : dict[str, list[tuple]]
        Mapping from OOV token -> contexts used (optional use/debug).
    """
    if role not in {"verb", "subject", "object"}:
        raise ValueError(f"role must be one of {{'verb','subject','object'}}, got {role!r}")

    # indices in the (v,s,o) triple
    r_idx = _role_index(role)
    other_idxs = [i for i in (0, 1, 2) if i != r_idx]

    # vocab keys inside tucker.vocab
    this_vocab_key = _voc_index(role)
    other_vocab_keys = [_voc_index("verb"), _voc_index("subject"), _voc_index("object")]
    other_vocab_keys = [other_vocab_keys[i] for i in other_idxs]

    this_vocab = tucker.vocab[this_vocab_key]
    other_vocabs = [tucker.vocab[k] for k in other_vocab_keys]

    # collect contexts where element in `role` is OOV but the other two are in-vocab
    out = defaultdict(list)
    for vector in sample:
        element = vector[r_idx]

        # already in vocab => skip
        if element in this_vocab:
            continue

        # other roles must be in vocab
        if all(vector[o_i] in o_v for o_i, o_v in zip(other_idxs, other_vocabs)):
            out[element].append(vector)


    # compute mean predicted role vector per OOV element
    extension = {}
    for element, vectors in out.items():
        reps = [tucker.predicted_role_vector(vector, role) for vector in vectors]
        extension[element] = np.mean(reps, axis=0)

    return extension

In [6]:
el_extension = extend_vocab("verb", sentence_sample, tucker)

In [7]:
mat_el = np.array(list(el_extension.values()))
verb_el = np.array(list(v_extension.values()))

np.allclose(mat_el, verb_el)

True

## Multi-threading the computations
Across different occurrences, we can easily parallelize across cpu nodes.


In [8]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
import numpy as np
from tqdm import tqdm
from similarity import get_eval_num_threads


from utils import ThreadBudget  # same as your eval code
# reuse your helper from the repo
# from <wherever> import get_eval_num_threads


def extend_vocab(
    role,
    sample,
    tucker,
    *,
    n_threads: int | None = None,
    thread_budget: ThreadBudget | None = None,
    fraction_threads: float = 0.75,
    min_threads: int = 1,
    min_count: int | None = None,
    top_k: int | None = None,
):
    """
    Build representations for OOV tokens in the given role ("verb", "subject", "object")
    by averaging tucker.predicted_role_vector(...) across all contexts where the other
    two roles are in-vocab.

    Threaded:
      - parallelizes predicted_role_vector calls with a ThreadPool
      - uses repo-style get_eval_num_threads() by default
      - optionally respects a ThreadBudget limiter (same pattern as evaluate_sample)

    Options
    -------
    min_count : int | None
        Keep only OOV tokens with >= min_count usable contexts.
    top_k : int | None
        Keep only the top_k OOV tokens by usable-context count (after min_count).
    """
    if role not in {"verb", "subject", "object"}:
        raise ValueError(f"role must be one of {{'verb','subject','object'}}, got {role!r}")

    # repo-style default threads
    if n_threads is None:
        n_threads = get_eval_num_threads(fraction=fraction_threads, min_threads=min_threads)

    # indices in the (v,s,o) triple
    r_idx = _role_index(role)
    other_idxs = [i for i in (0, 1, 2) if i != r_idx]

    # vocab keys inside tucker.vocab
    this_vocab_key = _voc_index(role)
    other_vocab_keys = [_voc_index("verb"), _voc_index("subject"), _voc_index("object")]
    other_vocab_keys = [other_vocab_keys[i] for i in other_idxs]

    this_vocab = tucker.vocab[this_vocab_key]
    other_vocabs = [tucker.vocab[k] for k in other_vocab_keys]

    # collect contexts where element in `role` is OOV but the other two are in-vocab
    out = defaultdict(list)
    for vector in tqdm(sample, desc="building OOV add list"):
        element = vector[r_idx]

        # already in vocab => skip
        if element in this_vocab:
            continue

        # other roles must be in vocab
        if all(vector[o_i] in o_v for o_i, o_v in zip(other_idxs, other_vocabs)):
            out[element].append(vector)

    # first check: if nothing at this point, stop computations
    if not out:
        return {}

    # filter by min_count / top_k (based on usable context count)
    if (min_count is not None) or (top_k is not None):
        counts = {tok: len(ctxs) for tok, ctxs in out.items()}

        if min_count is not None:
            keep = {tok for tok, c in counts.items() if c >= min_count}
        else:
            keep = set(counts.keys())

        if top_k is not None:
            # sort by count desc, then token for determinism
            ranked = sorted(keep, key=lambda t: (-counts[t], t))
            keep = set(ranked[:top_k])

        out = {tok: out[tok] for tok in keep}

    # second check: if nothing remains after filtering, do not proceed with computations
    if not out:
        return {}

    # ThreadBudget limiter (same pattern as evaluate_sample in similarity.py)
    limiter = (thread_budget.limit() if thread_budget is not None else None)
    if limiter is None:
        ctx = contextmanager(lambda: (yield))()  # no-op
    else:
        ctx = limiter

    # We accumulate sums+counts per token to avoid storing all reps together
    sums: dict[str, np.ndarray] = {}
    counts: dict[str, int] = {}

    def _one_call(tok, ctx_triple):
        rep = tucker.predicted_role_vector(ctx_triple, role)
        return tok, np.asarray(rep)

    # flatten jobs: 1 job per usable context
    jobs = [(tok, ctx_triple) for tok, ctxs in out.items() for ctx_triple in ctxs]
    if not jobs:
        return {}

    with ctx:
        with ThreadPoolExecutor(max_workers=n_threads) as ex:
            futures = [ex.submit(_one_call, tok, ctx_triple) for tok, ctx_triple in jobs]

            for fut in tqdm(as_completed(futures), total=len(futures), desc="calculating representations"):
                tok, rep = fut.result()

                if tok not in sums:
                    sums[tok] = rep.copy()
                    counts[tok] = 1
                else:
                    sums[tok] += rep
                    counts[tok] += 1

    extension = {tok: sums[tok] / counts[tok] for tok in sums.keys()}

    return extension


In [9]:
from similarity import load_og_sentences
full_dataset = load_og_sentences(vector_path=vector_path)

In [10]:
n_threads = get_eval_num_threads(0.50, 1)
extensions = {}
extensions["verb"] = extend_vocab("verb", full_dataset, tucker, n_threads=n_threads, min_count=50)
extensions["subject"] = extend_vocab("subject", full_dataset, tucker, n_threads=n_threads, min_count=50)
extensions["object"] = extend_vocab("object", full_dataset, tucker, n_threads=n_threads, min_count=50)
print(f"Added {len(extensions["verb"])} new verbs,\n{len(extensions["subject"])} new subjects,\n{len(extensions["subject"])} new objects")

calculating representations: 100%|██████████| 482308/482308 [01:18<00:00, 6161.37it/s]


Added 369 new verbs,
2475 new subjects,
2475 new objects


# Evaluation
We check if the extension actually works:
one of the added verbs (from inspection) is "guard". We inspect its dimensions, and check its similarity to semantically similar, as well as more distant verbs.

In [16]:
from contextlib import contextmanager
import numpy as np
import torch
from tqdm import tqdm
from utils import ThreadBudget
from similarity import get_eval_num_threads

def _safe_cosine(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)

    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)

    a_norm = max(a_norm, eps)
    b_norm = max(b_norm, eps)

    return float((a @ b) / (a_norm * b_norm))

def get_role_vector(role: str, token: str, tucker, extension: dict[str, np.ndarray] | None = None) -> np.ndarray:
    role2idx = {"verb": 0, "subject": 1, "object": 2}
    role2voc = {"verb": "vocab_v", "subject": "vocab_s", "object": "vocab_o"}
    role2i = {"verb": "v2i", "subject": "s2i", "object": "o2i"}

    if token in tucker.vocab[role2voc[role]]:
        idx = tucker.vocab[role2i[role]][token]
        # factor rows are the role embeddings (same objects used in evaluate_sample)
        return tucker.factors[role2idx[role]][idx].detach().cpu().numpy()

    if extension is not None and token in extension:
        return np.asarray(extension[token])

    raise KeyError(f"{token!r} not found in tucker vocab for role={role!r} and not in extension.")


def run_tests(element, role, targets=None, most_similar=False, include_extension=False):
    if not element in extensions[role].keys():
        raise KeyError(f"{element} was not added to {role} extension.")
    target_vec = extensions[role][element]
    results = {}
    role2voc = {"verb": "vocab_v", "subject": "vocab_s", "object": "vocab_o"}

    if targets:
        for w in targets:
            if w in tucker.vocab[role2voc[role]]:
                w_vec = get_role_vector(role, w, tucker, extension=extensions[role])
                results[w] = _safe_cosine(target_vec, w_vec)
            else:
                print(f"{w} not in vocab")
        if not results:
            raise KeyError(f"None of {targets} is in original vocab")

        print("Cosine similarity (verb space):")
        for w in results.keys():
            print(f"{element} vs {w:<7}: {results[w]:.4f}")

    if most_similar:
        k = 5
        i = _role_index(role)

        # original vocab factor matrix for this role: (N, R)
        F = tucker.factors[i].detach().cpu().numpy().astype(np.float64, copy=False)

        # --- defensive norm computation ---
        eps = 1e-12
        F_norm = np.maximum(np.linalg.norm(F, axis=1), eps)   # (N,)
        G_norm = max(np.linalg.norm(target_vec), eps)         # scalar

        # --- safe cosine similarities ---
        similarities = (F @ target_vec) / (F_norm * G_norm)   # (N,)

        # top-k from original vocab
        k_eff = min(k, similarities.shape[0])
        top_idx = np.argpartition(-similarities, k_eff - 1)[:k_eff]
        top_idx = top_idx[np.argsort(-similarities[top_idx])]

        vocab_tokens = tucker.vocab[f"vocab_{role[0]}"]  # assumes list-like idx -> token
        top_vocab = [(vocab_tokens[j], float(similarities[j])) for j in top_idx]

        print(f"\nTop {len(top_vocab)} most similar (original vocab) in {role} space to {element!r}:")
        for tok, sim in top_vocab:
            print(f"{tok:<20} {sim:.4f}")

        # separately: top-k among extension-only tokens
        if include_extension:
            ext_scores = []
            for tok, vec in extensions[role].items():
                if tok in tucker.vocab[role2voc[role]]:
                    continue  # skip tokens already present in original vocab
                if tok == element:
                    continue  # don't show the query token itself
                score = _safe_cosine(target_vec, np.asarray(vec, dtype=np.float64))
                ext_scores.append((tok, float(score)))

            ext_scores.sort(key=lambda x: x[1], reverse=True)
            top_ext = ext_scores[:k]

            print(f"\nTop {len(top_ext)} most similar (extension-only) in {role} space to {element!r}:")
            for tok, sim in top_ext:
                print(f"{tok:<20} {sim:.4f}")



In [17]:
guard = extensions["verb"]["guard"]

In [18]:
import torch
scores, indices = torch.topk(torch.tensor(guard), 20)
print("top dimensions for guard:")
for (ind, score) in zip(indices.cpu().numpy(), scores):
    print(ind, score)

top dimensions for guard:
97 tensor(850386.0625)
91 tensor(642303.5000)
95 tensor(473891.)
146 tensor(365156.3750)
14 tensor(339261.)
52 tensor(271480.8438)
26 tensor(212742.9688)
67 tensor(173267.4062)
106 tensor(172199.4531)
102 tensor(169899.1719)
100 tensor(140060.8594)
121 tensor(140058.0781)
130 tensor(134318.1250)
76 tensor(129960.3203)
120 tensor(123137.8281)
3 tensor(118042.5312)
63 tensor(117386.3203)
83 tensor(114188.3281)
68 tensor(113576.7031)
69 tensor(109430.5391)


In [19]:
guard_vec = extensions["verb"]["guard"]

targets = ["protect", "defend", "walk", "attack", "cry", "look"]
results = {}

for w in targets:
    w_vec = get_role_vector("verb", w, tucker, extension=extensions["verb"])
    results[w] = _safe_cosine(guard_vec, w_vec)

print("Cosine similarity (verb space):")
for w in targets:
    print(f"guard vs {w:<7}: {results[w]:.4f}")


Cosine similarity (verb space):
guard vs protect: 0.4507
guard vs defend : 0.6889
guard vs walk   : 0.3334
guard vs attack : 0.5720
guard vs cry    : 0.1249
guard vs look   : 0.1792


In [20]:
run_tests("guard", "verb", targets, most_similar=True)

Cosine similarity (verb space):
guard vs protect: 0.4507
guard vs defend : 0.6889
guard vs walk   : 0.3334
guard vs attack : 0.5720
guard vs cry    : 0.1249
guard vs look   : 0.1792

Top 5 most similar (original vocab) in verb space to 'guard':
beat                 0.7884
treat                0.7309
blame                0.7116
forgive              0.7051
catch                0.6994


In [21]:
run_tests("guard", "verb", targets, most_similar=True, include_extension=True)

Cosine similarity (verb space):
guard vs protect: 0.4507
guard vs defend : 0.6889
guard vs walk   : 0.3334
guard vs attack : 0.5720
guard vs cry    : 0.1249
guard vs look   : 0.1792

Top 5 most similar (original vocab) in verb space to 'guard':
beat                 0.7884
treat                0.7309
blame                0.7116
forgive              0.7051
catch                0.6994

Top 5 most similar (extension-only) in verb space to 'guard':
spoil                0.9819
hook                 0.9762
crush                0.9759
chase                0.9743
wave                 0.9710


In [22]:
targets = ["travel", "walk", "climb", "cry", "surprise", "look", "defend", "protect", "attack", "rise", "abandon"]
run_tests("hike", "verb", targets, most_similar=True)

Cosine similarity (verb space):
hike vs travel : 0.1482
hike vs walk   : 0.2505
hike vs climb  : 0.1726
hike vs cry    : 0.1239
hike vs surprise: 0.1685
hike vs look   : 0.2207
hike vs defend : 0.4725
hike vs protect: 0.3170
hike vs attack : 0.3688
hike vs rise   : 0.0252
hike vs abandon: 0.4525

Top 5 most similar (original vocab) in verb space to 'hike':
catch                0.6653
fix                  0.6638
own                  0.6582
spot                 0.6488
beat                 0.6328


In [23]:
run_tests("shock", "verb", targets, most_similar=True, include_extension=True)

Cosine similarity (verb space):
shock vs travel : 0.0586
shock vs walk   : 0.2872
shock vs climb  : 0.0379
shock vs cry    : 0.0484
shock vs surprise: 0.7983
shock vs look   : 0.1063
shock vs defend : 0.5552
shock vs protect: 0.4584
shock vs attack : 0.4107
shock vs rise   : 0.0128
shock vs abandon: 0.4200

Top 5 most similar (original vocab) in verb space to 'shock':
convince             0.8420
surprise             0.7983
force                0.7962
prompt               0.7182
inspire              0.7179

Top 5 most similar (extension-only) in verb space to 'shock':
compel               0.9871
upset                0.9810
haunt                0.9803
interest             0.9776
amaze                0.9744


In [24]:
run_tests("surrender", "verb", targets, most_similar=True, include_extension=True)

Cosine similarity (verb space):
surrender vs travel : 0.1701
surrender vs walk   : 0.2838
surrender vs climb  : 0.1773
surrender vs cry    : 0.1412
surrender vs surprise: 0.2143
surrender vs look   : 0.2141
surrender vs defend : 0.5280
surrender vs protect: 0.2969
surrender vs attack : 0.4567
surrender vs rise   : 0.0267
surrender vs abandon: 0.4988

Top 5 most similar (original vocab) in verb space to 'surrender':
beat                 0.6912
catch                0.6837
judge                0.6641
approach             0.6580
hand                 0.6523

Top 5 most similar (extension-only) in verb space to 'surrender':
squeeze              0.9799
battle               0.9762
film                 0.9750
pitch                0.9720
delegate             0.9709


In [29]:
extensions["object"].keys()

dict_keys(['k', 'architecture', 'worry', 'burn', 'cheer', 'Prize', 'toilet', 'organizer', 'river', 'illness', 'apology', 'diagnosis', 'bandwidth', 'scholarship', 'tenant', 'count', 'stamp', 'negotiation', 'trait', 'wiring', 'tournament', 'accolade', 'dominance', 'revolution', 'compromise', 'odor', 'iPhone', 'badge', 'butter', 'myth', 'shock', 'sword', 'genre', 'richness', 'token', 'dignity', 'tweet', 'assembly', 'disclosure', 'cuisine', 'proceeding', 'misunderstanding', 'attire', 'Application', 'football', 'gram', 'surveillance', 'rope', 'manuscript', 'bass', 'Lord', 'discovery', 'mindfulness', 'streaming', 'poverty', 'shame', 'composition', 'courage', 'shipment', 'clarification', 'newsletter', 'inclination', 'silhouette', 'like', 'aircraft', 'the', 'taco', 'achievement', 'participation', 'Centre', 'receiver', 'waiver', 'punishment', 'illustration', 'graphic', 'current', 'velocity', 'producer', 'resolve', 'archive', 'closet', 'timer', 'trademark', 'goodness', 'hairstyle', 'sanctuary', 

In [27]:
run_tests("Walmart", "subject", most_similar=True, include_extension=True)


Top 5 most similar (original vocab) in subject space to 'Walmart':
shop                 0.6646
community            0.6629
hotel                0.6502
firm                 0.6454
brand                0.6450

Top 5 most similar (extension-only) in subject space to 'Walmart':
Vegas                0.9872
Store                0.9843
Available            0.9816
UAE                  0.9796
Library              0.9790


In [28]:
run_tests("laboratory", "subject", most_similar=True, include_extension=True)


Top 5 most similar (original vocab) in subject space to 'laboratory':
brand                0.6795
firm                 0.6699
organization         0.6464
community            0.6237
provider             0.6226

Top 5 most similar (extension-only) in subject space to 'laboratory':
lab                  0.9854
Assistant            0.9789
factory              0.9764
Technologies         0.9759
Industries           0.9739


In [32]:
run_tests("horror", "object", most_similar=True, include_extension=True)


Top 5 most similar (original vocab) in object space to 'horror':
face                 0.7475
shot                 0.7219
character            0.7137
episode              0.6884
band                 0.6847

Top 5 most similar (extension-only) in object space to 'horror':
mystery              0.9897
Christmas            0.9880
inside               0.9871
shadow               0.9818
America              0.9812


In [33]:
run_tests("repository", "object", most_similar=True, include_extension=True)


Top 5 most similar (original vocab) in object space to 'repository':
update               0.7056
mix                  0.6946
trial                0.6915
price                0.6879
session              0.6857

Top 5 most similar (extension-only) in object space to 'repository':
tier                 0.9871
assortment           0.9808
graphic              0.9799
portal               0.9794
gateway              0.9794


In [34]:
run_tests("lunch", "object", most_similar=True, include_extension=True)



Top 5 most similar (original vocab) in object space to 'lunch':
night                0.7056
shot                 0.6968
suggestion           0.6960
session              0.6927
spot                 0.6879

Top 5 most similar (extension-only) in object space to 'lunch':
dinner               0.9933
pitch                0.9898
breakfast            0.9887
snack                0.9885
treat                0.9879
